# Lab 046 — FT-Transformer: feature tokenizer + [CLS] readout

**Lesson:** [`lessons/0046-ft-transformer.html`](../lessons/0046-ft-transformer.html) · **Phase / Year:** Year 2 · Q1

**Paper:** Gorishniy, Rubachev, Khrulkov & Babenko 2021, *Revisiting Deep Learning Models for Tabular Data* ([arXiv:2106.11959](https://arxiv.org/abs/2106.11959)) — §3.3 (the Feature Tokenizer, the [CLS] token, the Transformer), §5 (the fair-comparison result: FT-Transformer is the top deep model and ~ties tuned GBDTs). Self-attention: Vaswani et al. 2017 ([arXiv:1706.03762](https://arxiv.org/abs/1706.03762)). Predecessor it fixes: [L045 TabTransformer](../lessons/0045-tabtransformer.html) (numeric-bypass).

**Dataset tier:** **A** — small real OpenML tables via `relkit` (CPU-cheap; deliberately NOT representative — see the #23 notes in Task 3).

**Skill you are practising:** build FT-Transformer's two load-bearing pieces — (1) the **numeric tokenizer** `T_j = b_j + x_j·W_j` and (2) the **[CLS]** readout — then race FT-Transformer vs TabTransformer vs MLP vs CatBoost, and prove the numeric-bypass fix with a direct probe. The model you train calls *your* Task-1/Task-2 functions (they are inlined-and-kept, not hidden behind a package).

**Exit criteria:** EXIT TICKET prints your tokenizer + [CLS] validation, the bake-off ranks, the FT-T-vs-TabTransformer count, the numerics-attend probe, and one sentence — *what does tokenising numerics buy, and does it beat a tree?*

---

### How this notebook works
- **PROVIDED** cells — boilerplate (data, frames, budgets, baselines) **and** the paper's FT-Transformer copied into the notebook (not hidden behind `import relkit.ft_transformer`); just run.
- **TODO** cells — blanks (`____`); you implement the skill.
- **CHECK** cells — immediate feedback; do not edit.
- Run top to bottom. After EXIT, a **NEXT STEP** cell trains closer to the paper (Colab GPU or Modal). When **EXIT TICKET** prints cleanly, paste it to your teacher or say *"lab done"*.

### Environment
One-time: `bash labs/setup-env.sh` → kernel **Relational Labs (.venv)**. Needs **torch** + scikit-learn + **catboost** (CPU is fine). Real datasets fetch from OpenML on first run then cache. Budget: **~8–12 minutes on CPU** — set `OMP_NUM_THREADS=1` if a run feels slow (thread oversubscription has been the real cause of every slow lab). This lab uses a deliberately small budget/seed/dataset count to stay interactive; the lesson's headline numbers come from the fuller `labs/_verify_l046.py` run plus the paper's 11-dataset tuned benchmark.

### Running on Google Colab?

Colab opens only this single file, so the course package (`relkit`) and the lab
dependencies (xgboost, lightgbm, catboost, …) are **not** present by default. The cell
below fixes that: on Colab it shallow-clones the course repo, installs
`requirements-labs.txt`, and switches into `labs/` so `relkit` imports and the data cache
resolve. **On a local venv or your own Jupyter it does nothing — just run it and continue.**

In [ ]:
# @colab-bootstrap — PROVIDED. Makes the lab self-sufficient on Google Colab; a no-op elsewhere.
import os, sys

if "google.colab" in sys.modules:
    if not os.path.isdir("/content/relational"):
        !git clone --depth 1 https://github.com/Avistian/relational.git /content/relational
    %pip install -q -r /content/relational/requirements-labs.txt
    os.chdir("/content/relational/labs")
    print("Colab ready — working dir:", os.getcwd())
else:
    print("Not on Colab — using the local environment as-is.")

## Concept recap — FT-Transformer = Feature Tokenizer + Transformer

**The one idea.** A Transformer eats a sequence of **tokens** (vectors of a common width *d*). [TabTransformer](../lessons/0045-tabtransformer.html)
only knew how to make tokens from *categories* (an embedding lookup), so its **numeric** features bypassed
the attention — LayerNorm'd and concatenated after the Transformer, never contextualised. FT-Transformer's
**Feature Tokenizer** adds the missing half: a way to turn a *number* into a token.

**The tokenizer (Gorishniy 2021, §3.3).**
- **Numeric feature *j*:** `T_j = b_j + x_j · W_j` — an **affine** embedding of the scalar onto a learned
  direction `W_j` (with `W_j, b_j ∈ ℝ^d`). Order is preserved: 39 and 40 land near each other. *(Task 1.)*
- **Categorical feature *j*:** `T_j = b_j + e_j[x_j]` — the [L031](../lessons/0031-embeddings-for-categoricals.html)
  entity embedding plus a per-feature bias.

**The [CLS] readout.** Prepend one learned `[CLS]` token (representing no feature) at position 0. Through the
Transformer it *collects* information from every feature token, and its final vector is the row summary the
head reads. *(Task 2.)*

**Why it matters.** Because numerics are now tokens, a numeric feature both attends and is attended to — so a
change in `age` can reshape the whole row's [CLS] summary (Task 4 measures this: FT-T moves, TabTransformer
moves 0). That fix makes FT-Transformer the **strongest single neural baseline** — beating TabTransformer
most where numerics carry the signal — while still, honestly, losing the flat-table metric to CatBoost.

Full write-up + the tokenizer and architecture widgets: [Lesson 046](../lessons/0046-ft-transformer.html).

## Setup — PROVIDED (tables + shared frames + baselines + budgets)

In [ ]:
# PROVIDED — imports, tables, the shared frame, and budgets. Just run.
# The FT-Transformer *model* is inlined in a later cell so you can read it (Gorishniy 2021, §3.3, Fig. 2).
# `relkit.ft_transformer` here provides only the two Task checkers; the baselines (TabTransformer from L045,
# MLP from L042) are imported for the bake-off — the model being TAUGHT (FT-T) is the one you build (NOTES #25).
import warnings
warnings.filterwarnings("ignore")

import time
import numpy as np
import torch
from scipy.stats import rankdata, friedmanchisquare
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.preprocessing import OneHotEncoder, StandardScaler

import sys
from pathlib import Path
sys.path.insert(0, str(Path(".").resolve()))          # labs/ when run from there
sys.path.insert(0, str(Path(".").resolve().parent))   # labs/ when run from labs/solutions/
from relkit import load_tier_a
from relkit.tabtransformer import (
    frame_categorical,                                         # data helper (not the model)
    TabTransformer, train_tabtransformer, tabtransformer_auc,  # L045 baseline (numeric-bypass)
)
from relkit.nets import TabMLP, train_net, net_auc             # L042 baseline (strong deep model)
from relkit.ft_transformer import (
    affine_numeric_tokens as relkit_affine,                    # checker only (NOTES #22)
    prepend_cls as relkit_prepend,                             # checker only (NOTES #22)
)

DEVICE = "cpu"
DATASETS  = ["adult", "churn"]        # a mixed table + a numeric-heavy one, to show the numeric-bypass fix
SUBSAMPLE = {"adult": 3000}           # keep CPU cost bounded — a down-scaled demonstration (#20/#23)
SEEDS = [0, 1]                        # smaller than the lesson's 4-table/3-seed run, to stay interactive
EPOCHS, PATIENCE, BS, LR = 50, 10, 256, 1e-3
FT_CFG = dict(d=64, n_layers=3, n_heads=8, dropout=0.1)                    # FT-Transformer
TT_CFG = dict(d=32, n_layers=3, n_heads=4, head_hidden=128, dropout=0.1)   # TabTransformer (L045)

def load_frame(name, cap=None):
    """Encode a table into three shared frames: integer cats + scaled nums (transformers), a dense one-hot
    matrix (MLP), and the raw DataFrame (CatBoost native categoricals). `cap` bounds rows for CPU speed."""
    Xdf, y = load_tier_a(name)
    cap = cap if cap is not None else SUBSAMPLE.get(name)
    if cap and len(Xdf) > cap:
        idx, _ = train_test_split(np.arange(len(Xdf)), train_size=cap, random_state=0, stratify=y)
        Xdf, y = Xdf.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    Xcat, Xnum, cards, cat_names, num_names = frame_categorical(Xdf)
    ct = ColumnTransformer([("num", StandardScaler(), num_names),
                            ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_names)])
    Xdense = ct.fit_transform(Xdf).astype(np.float32)
    return {"Xdf": Xdf, "y": y.to_numpy().astype(np.float32), "Xcat": Xcat, "Xnum": Xnum, "cards": cards,
            "cat_names": cat_names, "num_names": num_names, "Xdense": Xdense,
            "num_frac": len(num_names) / max(len(num_names) + len(cat_names), 1)}

def split_idx(n, y, seed):
    """The SHARED frame — identical train/val/test for every arm (L020 contract, L042 protocol)."""
    tr, te = train_test_split(np.arange(n), test_size=0.30, random_state=seed, stratify=y)
    tr, va = train_test_split(tr, test_size=0.25, random_state=seed, stratify=y[tr])
    return tr, va, te

print("setup ok — torch", torch.__version__)

## Task 1 — `affine_numeric_tokens`: turn a NUMBER into a token

**Goal.** Write the numeric half of FT-Transformer's Feature Tokenizer. Each numeric feature *j* gets its
own learned weight vector `W_j` and bias `b_j` (both of width *d*), and its token is the **affine** map

$$T_j = b_j + x_j \cdot W_j$$

**Why affine, not a lookup.** TabTransformer only knew how to tokenise *categories* (an embedding lookup),
so its numerics bypassed the attention entirely. Embedding a number as `b_j + x_j·W_j` places the scalar on
a learned direction `W_j`, preserving order (39 and 40 land near each other) — so a *number* becomes a
first-class token that can attend. This one function is the whole idea of the lesson.

**Worked micro-example.** With `W_j = [2, 0]`, `b_j = [1, 5]`, and `x_j = 3`: the token is
`[1, 5] + 3·[2, 0] = [7, 5]`. Bump `x_j` to 4 and it becomes `[9, 5]` — moved by `Δx·W_j = [2, 0]`.

In [ ]:
# TODO — implement the numeric tokenizer. Fill every ____.
def affine_numeric_tokens(x_num, weight, bias):
    """x_num: [B, n_num] float · weight, bias: [n_num, d] · returns tokens [B, n_num, d].
    Each numeric feature j -> token_j = bias_j + x_num[:, j] * weight_j."""
    # broadcast the scalar over the d embedding dims: [B, n_num, 1] * [n_num, d] -> [B, n_num, d]
    return ____

# quick look on the worked micro-example
W = torch.tensor([[2., 0.]]); b = torch.tensor([[1., 5.]])   # one numeric feature, d=2
x = torch.tensor([[3.], [4.]])                               # two rows
print("tokens:\n", affine_numeric_tokens(x, W, b).squeeze(1).numpy())  # -> [[7,5],[9,5]]

In [ ]:
# CHECK — the numeric tokenizer is affine and per-feature (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

torch.manual_seed(0)
B, n_num, d = 16, 5, 8
Xn = torch.randn(B, n_num)
Wt = torch.randn(n_num, d); bs = torch.randn(n_num, d)
T = affine_numeric_tokens(Xn, Wt, bs)
chk("shape is [B, n_num, d]", tuple(T.shape) == (B, n_num, d), f"{tuple(T.shape)}")

# affine: bumping feature j by Δ moves token j by exactly Δ·W_j, and moves NO other token
j, delta = 2, 2.5
Xn2 = Xn.clone(); Xn2[:, j] += delta
dT = affine_numeric_tokens(Xn2, Wt, bs) - T
chk("token j moves by exactly Δx·W_j (affine in the scalar)",
    torch.allclose(dT[:, j], (delta * Wt[j]).expand(B, d), atol=1e-5))
others = dT.clone(); others[:, j] = 0.0
chk("changing feature j leaves every OTHER token unchanged", others.abs().max().item() < 1e-6)

# VALIDATION against the from-scratch reference (NOTES #22)
chk("VALIDATED against relkit.ft_transformer.affine_numeric_tokens",
    torch.allclose(affine_numeric_tokens(Xn, Wt, bs), relkit_affine(Xn, Wt, bs), atol=1e-6))
print("\nTask 1", "OK" if ok else "-- fix the FAILs above")

## Task 2 — `prepend_cls`: the learned [CLS] readout token

**Goal.** Prepend a single learned `[CLS]` token at position 0 of the token sequence. It represents **no
feature**; its job is to *collect* information from all the feature tokens through attention, and after the
Transformer its final vector is the row summary the head reads out (the BERT trick).

**Shapes.** `tokens` is `[B, k, d]` (the *k* feature tokens); `cls` is a single learned `[1, 1, d]` vector.
You expand `cls` across the batch and concatenate it in front, giving `[B, k+1, d]` with `[CLS]` at index 0.

**Why a dedicated token.** You *could* average the feature tokens, but a learned [CLS] lets the model choose,
through attention, how much to read from each feature — a strictly more flexible pool, and the one the paper
uses.

In [ ]:
# TODO — prepend the [CLS] token. Fill every ____.
def prepend_cls(tokens, cls):
    """tokens: [B, k, d] · cls: [1, 1, d] (a single learned vector) · returns [B, k+1, d]."""
    B = tokens.shape[0]
    # expand the one learned cls vector across the batch, then concatenate it at position 0
    cls_batch = cls.expand(B, 1, cls.shape[-1])
    return ____

# quick look
toks = torch.zeros(4, 3, 8)                       # 4 rows, 3 feature tokens, d=8
cls = torch.arange(8, dtype=torch.float32).reshape(1, 1, 8)
seq = prepend_cls(toks, cls)
print("sequence shape:", tuple(seq.shape), " (should be [4, 4, 8])")
print("position 0 == the cls vector for every row:", bool(torch.allclose(seq[:, 0], cls.reshape(1, 8).expand(4, 8))))

In [ ]:
# CHECK — [CLS] is prepended correctly (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

torch.manual_seed(0)
toks = torch.randn(6, 5, 8); cls = torch.randn(1, 1, 8)
seq = prepend_cls(toks, cls)
chk("sequence grew by one token: [B, k+1, d]", tuple(seq.shape) == (6, 6, 8), f"{tuple(seq.shape)}")
chk("position 0 is the SAME learned [CLS] for every row", torch.allclose(seq[:, 0], cls.reshape(1, 8).expand(6, 8)))
chk("positions 1..k are the original feature tokens, unchanged", torch.allclose(seq[:, 1:], toks))
chk("VALIDATED against relkit.ft_transformer.prepend_cls", torch.allclose(seq, relkit_prepend(toks, cls)))
print("\nTask 2", "OK" if ok else "-- fix the FAILs above")

## The rest of the paper's architecture — inlined, not imported

The next cell is `labs/relkit/ft_transformer.py` copied into this notebook so you can **read every line** of the Feature Tokenizer, the [CLS] token, the PreNorm Transformer block, and the train / eval loops. It is not `from relkit import ...` hiding a model behind a package. `affine_numeric_tokens` and `prepend_cls` defined in your TODO cells above are **kept** — this copy skips those names, so the encoder you train next calls *your* functions.

`relkit/` still holds the canonical file for Modal / `_verify` so the two cannot drift in opposite directions; the notebook is a readable copy, not a black box.

In [ ]:
# PROVIDED — inlined from `labs/relkit/ft_transformer.py` so you can read every line.
# This is the paper implementation, not `import relkit...` hiding it.
# Canonical file stays at labs/relkit/ft_transformer.py for Modal / _verify; this cell is a copy.

"""From-scratch FT-Transformer (Gorishniy, Rubachev, Khrulkov & Babenko 2021, arXiv:2106.11959) — Lesson 046.

Built from the paper (Fig. 2 + §3.3), not from a library (NOTES standards #18/#22/#24). rtdl's own
`FTTransformer` and torch's `nn.MultiheadAttention` / `F.scaled_dot_product_attention` are VALIDATION
points only (`labs/_check_l046.py`, `labs/_verify_l046.py`) — never imported here to *build* the model.

FT-Transformer = **Feature Tokenizer + Transformer**. The one idea that separates it from TabTransformer
(L045): *every* feature becomes a token — **numeric features too** — so numbers finally attend. A learned
`[CLS]` token is appended; after the Transformer stack its final embedding is the row representation the
head reads. This removes TabTransformer's numeric-bypass limitation (L045's headline gap).

Paper map — every piece cites the element it realises (Gorishniy 2021, §3.3 "FT-Transformer",
Fig. 2 left = Feature Tokenizer, Fig. 2 right = Transformer + [CLS] readout):

| Paper                                                                     | Here                    |
|---------------------------------------------------------------------------|-------------------------|
| numeric feature j -> token  T_j = b_j + x_j · W_j   (W_j, b_j in R^d)      | `FeatureTokenizer.num_*`|
| categorical feature j -> token  T_j = b_j + e_j[x_j]  (per-column embed)   | `FeatureTokenizer.cat_*`|
| stack the k feature tokens -> [B, k, d]                                    | `FeatureTokenizer.forward`|
| prepend a learned [CLS] token -> [B, k+1, d]                              | `FTTransformer.cls`     |
| L Transformer layers, **PreNorm** (LN before each sub-layer)              | `FTTransformerBlock`    |
| readout: take the final [CLS] token -> LN -> ReLU -> Linear -> 1 logit    | `FTTransformer.forward` |
| multi-head self-attention  softmax(QKᵀ/√d)V per head, concat, project      | reused `MultiHeadSelfAttention` |

Fidelity notes (stated so the student is not misled, NOTES #20): the paper uses a PreNorm block, skips the
very first LayerNorm (a detail that matters little at our scale), and uses a ReGLU FFN; we use a plain
GELU FFN. The load-bearing ideas — per-feature tokenization *including numerics*, the [CLS] readout, and
PreNorm depth — are reproduced exactly. Binary classification only (one logit). CPU is fine for Tier-A.
"""
from __future__ import annotations

import numpy as np
import torch
import torch.nn as nn
from sklearn.metrics import roc_auc_score

# Reuse the from-scratch attention validated in L045 (machine-precision vs torch in _check_l045).
# This is not "importing a library model" — it is our own paper-built kernel promoted to relkit (#22).
from relkit.tabtransformer import MultiHeadSelfAttention, frame_categorical  # noqa: F401 (re-exported)


# ---------------------------------------------------------------- the two load-bearing tokenizer ops
# These are module-level so the lab notebook can inline the rest of this file while KEEPING the student's
# own versions (the lab writes exactly these two functions; standard #25 — the paper's code is visible AND
# the student's code is the one that runs).
# ---------------------------------------------------------------- Feature Tokenizer (Gorishniy §3.3, Fig. 2 left)
class FeatureTokenizer(nn.Module):
    """Turn every feature — numeric AND categorical — into a d-dim token (Gorishniy 2021, §3.3).

    numeric  T_j = b_j + x_j · W_j         (W_j, b_j learned d-vectors; the token is AFFINE in x_j)
    category T_j = b_j + e_j[x_j]          (per-column embedding table + a per-column bias)

    Output: [B, k, d] with k = n_num + n_cat tokens. This is the whole difference from TabTransformer,
    which never tokenises numerics — here a numeric feature is a first-class token that attends and is
    attended to.
    """

    def __init__(self, n_num: int, cards, d: int):
        super().__init__()
        self.n_num, self.cards, self.d = n_num, list(cards), d
        self.n_cat = len(self.cards)
        # numeric: one learned weight + bias vector per numeric feature (element-wise linear -> d dims)
        self.num_weight = nn.Parameter(torch.empty(n_num, d)) if n_num > 0 else None
        self.num_bias = nn.Parameter(torch.empty(n_num, d)) if n_num > 0 else None
        # categorical: an embedding table + a learned bias per categorical feature
        self.cat_embs = nn.ModuleList([nn.Embedding(max(c, 1), d) for c in self.cards])
        self.cat_bias = nn.Parameter(torch.empty(self.n_cat, d)) if self.n_cat > 0 else None
        self.reset_parameters()

    def reset_parameters(self):
        # Small uniform init (the paper's Kaiming-uniform-style start; the exact scheme is not load-bearing).
        bound = 1.0 / (self.d ** 0.5)
        if self.num_weight is not None:
            nn.init.uniform_(self.num_weight, -bound, bound)
            nn.init.uniform_(self.num_bias, -bound, bound)
        for emb in self.cat_embs:
            nn.init.uniform_(emb.weight, -bound, bound)
        if self.cat_bias is not None:
            nn.init.uniform_(self.cat_bias, -bound, bound)

    def forward(self, x_num, x_cat):
        """x_num: [B, n_num] float, x_cat: [B, n_cat] long. Returns tokens [B, k, d]."""
        B = x_num.shape[0] if self.n_num > 0 else x_cat.shape[0]
        tokens = []
        if self.n_num > 0:
            # broadcast: [B, n_num, 1] * [n_num, d] -> [B, n_num, d], then add the per-feature bias
            num_tok = affine_numeric_tokens(x_num, self.num_weight, self.num_bias)
            tokens.append(num_tok)
        if self.n_cat > 0:
            cat_tok = torch.stack(
                [emb(x_cat[:, j]) for j, emb in enumerate(self.cat_embs)], dim=1
            ) + self.cat_bias
            tokens.append(cat_tok)
        if not tokens:
            return torch.zeros(B, 0, self.d, device=x_cat.device)
        return torch.cat(tokens, dim=1)                        # [B, k, d]


# ---------------------------------------------------------------- Transformer block (PreNorm, Gorishniy §3.3)
class FTTransformerBlock(nn.Module):
    """One PreNorm Transformer layer (Gorishniy 2021 uses PreNorm, unlike TabTransformer's PostNorm):
    LayerNorm BEFORE each sub-layer, residual AROUND it. PreNorm keeps deeper stacks trainable — the
    residual carries the identity path, each sub-layer only adds a correction (the L028 skip idea)."""

    def __init__(self, d, n_heads=8, ff_hidden=None, dropout=0.1):
        super().__init__()
        ff_hidden = ff_hidden or int(d * 4 / 3)              # paper's ~4/3 ratio (ReGLU); we use GELU
        self.norm1 = nn.LayerNorm(d)
        self.attn = MultiHeadSelfAttention(d, n_heads)
        self.norm2 = nn.LayerNorm(d)
        self.ffn = nn.Sequential(nn.Linear(d, ff_hidden), nn.GELU(), nn.Dropout(dropout),
                                 nn.Linear(ff_hidden, d))
        self.drop = nn.Dropout(dropout)

    def forward(self, x):
        a, w = self.attn(self.norm1(x))                      # PreNorm: normalise, then attend
        x = x + self.drop(a)                                 # residual around attention
        f = self.ffn(self.norm2(x))
        x = x + self.drop(f)                                 # residual around the FFN
        return x, w


# ---------------------------------------------------------------- FT-Transformer (Gorishniy §3.3, Fig. 2)
class FTTransformer(nn.Module):
    """FT-Transformer = Feature Tokenizer + [CLS] + Transformer + [CLS]-readout head (Gorishniy 2021).

    Flow: tokenize every feature (numeric + categorical) -> prepend a learned [CLS] token -> L PreNorm
    Transformer blocks -> read the FINAL [CLS] token -> LayerNorm -> ReLU -> Linear -> 1 logit.

    Because numerics are tokens, a numeric feature both attends and is attended to — the exact capability
    TabTransformer (L045) lacked. n_layers=0 degenerates to "just the [CLS] readout of the raw tokens"
    (no attention), a useful ablation but NOT a standard baseline.
    """

    def __init__(self, n_num, cards, d=64, n_layers=3, n_heads=8, ff_hidden=None,
                 head_hidden=None, dropout=0.1):
        super().__init__()
        self.n_num, self.cards, self.d, self.n_layers = n_num, list(cards), d, n_layers
        self.k = n_num + len(self.cards)
        self.tokenizer = FeatureTokenizer(n_num, cards, d)
        self.cls = nn.Parameter(torch.empty(1, 1, d))
        nn.init.uniform_(self.cls, -1.0 / (d ** 0.5), 1.0 / (d ** 0.5))
        self.blocks = nn.ModuleList([FTTransformerBlock(d, n_heads, ff_hidden, dropout)
                                     for _ in range(n_layers)])
        self.head_norm = nn.LayerNorm(d)
        self.head_act = nn.ReLU()
        self.head = nn.Linear(d, 1)

    def tokens_with_cls(self, x_num, x_cat):
        """Feature tokens with the [CLS] token prepended: [B, k+1, d]. Position 0 is always [CLS]."""
        tok = self.tokenizer(x_num, x_cat)                   # [B, k, d]
        return prepend_cls(tok, self.cls)                    # [B, k+1, d], [CLS] at position 0

    def cls_readout(self, x_num, x_cat):
        """Run the Transformer and return the FINAL [CLS] embedding — the row representation (Fig. 2)."""
        h = self.tokens_with_cls(x_num, x_cat)
        for blk in self.blocks:
            h, _ = blk(h)
        return h[:, 0]                                        # the [CLS] token, position 0

    def forward(self, x_num, x_cat):
        z = self.cls_readout(x_num, x_cat)
        return self.head(self.head_act(self.head_norm(z))).squeeze(-1)


# ---------------------------------------------------------------- supervised training (fair protocol)
def _to_tensors(Xnum, Xcat, y=None, device="cpu"):
    xn = Xnum.to(device) if torch.is_tensor(Xnum) else torch.as_tensor(np.asarray(Xnum), dtype=torch.float32, device=device)
    xc = Xcat.to(device) if torch.is_tensor(Xcat) else torch.as_tensor(np.asarray(Xcat), dtype=torch.long, device=device)
    if y is None:
        return xn, xc
    yt = torch.as_tensor(np.asarray(y), dtype=torch.float32, device=device)
    return xn, xc, yt


def train_ft_transformer(model, Xnum_tr, Xcat_tr, ytr, Xnum_va, Xcat_va, yva, *,
                         lr=1e-3, wd=1e-5, max_epochs=80, patience=12, batch_size=256,
                         device="cpu", seed=0):
    """Mini-batch AdamW with early stopping on validation ROC-AUC — the same fair, shared-protocol
    contract as `relkit.nets.train_net` (L042) and `relkit.tabtransformer.train_tabtransformer` (L045):
    every arm picks its own training length by validation, so none is under- or over-trained.

    Returns (model_with_best_val_weights, best_val_auc).
    """
    torch.manual_seed(seed)
    model = model.to(device)
    xn, xc, yt = _to_tensors(Xnum_tr, Xcat_tr, ytr, device)
    xnv, xcv = _to_tensors(Xnum_va, Xcat_va, device=device)
    n = xn.shape[0]
    bs = min(batch_size, n)
    opt = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=wd)
    lossf = nn.BCEWithLogitsLoss()
    gen = torch.Generator().manual_seed(seed)
    best_auc, best_state, since = -1.0, None, 0
    for _ in range(max_epochs):
        model.train()
        perm = torch.randperm(n, generator=gen)
        for start in range(0, n, bs):
            idx = perm[start:start + bs]
            if idx.numel() < 2:
                continue
            opt.zero_grad()
            loss = lossf(model(xn[idx], xc[idx]), yt[idx])
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            pv = torch.sigmoid(model(xnv, xcv)).cpu().numpy()
        val_auc = roc_auc_score(yva, pv)
        if val_auc > best_auc:
            best_auc, since = val_auc, 0
            best_state = {k: v.detach().clone() for k, v in model.state_dict().items()}
        else:
            since += 1
            if since >= patience:
                break
    if best_state is not None:
        model.load_state_dict(best_state)
    return model, best_auc


@torch.no_grad()
def ft_transformer_auc(model, Xnum, Xcat, y, *, device="cpu"):
    model.eval()
    xn, xc = _to_tensors(Xnum, Xcat, device=device)
    return roc_auc_score(y, torch.sigmoid(model(xn, xc)).cpu().numpy())


## Task 3 — the bake-off: FT-Transformer vs TabTransformer vs MLP vs CatBoost

**Goal.** Race four models under one shared frame, then summarise with **mean ranks** and a Friedman test:

- **FT-Transformer** — tokenises *every* feature (your Tasks 1–2 are its load-bearing pieces).
- **TabTransformer** — the L045 model; numerics bypass the attention.
- **MLP** — the strong deep baseline ([L042](../lessons/0042-mlp-resnet-baselines.html)).
- **CatBoost** — the honest tree bar ([L042](../lessons/0042-mlp-resnet-baselines.html)'s baseline-first rule).

**What to look for (NOTES #23/#24).** Two readings, kept apart: (1) does FT-T beat TabTransformer, and is
the gap biggest where numerics dominate (`churn` has num-frac 0.80)? (2) is FT-T the best *neural* model,
while still losing to the tree? Two tables is a demonstration; the strong claim stays cited to the paper.

In [ ]:
# TODO — fill every ____.  (FTTransformer / train_ft_transformer / ft_transformer_auc are inlined below.)
def run_ft(fr, tr, va, te, seed):
    torch.manual_seed(seed)
    # build the FT-Transformer: it takes (n_num, cards, **cfg) — numerics AND categoricals become tokens
    m = FTTransformer(fr["Xnum"].shape[1], fr["cards"], **FT_CFG)
    m, _ = train_ft_transformer(m, fr["Xnum"][tr], fr["Xcat"][tr], fr["y"][tr],
                                fr["Xnum"][va], fr["Xcat"][va], fr["y"][va],
                                lr=LR, max_epochs=EPOCHS, patience=PATIENCE, batch_size=BS, seed=seed)
    return ft_transformer_auc(m, fr["Xnum"][te], fr["Xcat"][te], fr["y"][te])

def run_tabt(fr, tr, va, te, seed):
    torch.manual_seed(seed)
    m = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **TT_CFG)
    m, _ = train_tabtransformer(m, fr["Xcat"][tr], fr["Xnum"][tr], fr["y"][tr],
                                fr["Xcat"][va], fr["Xnum"][va], fr["y"][va],
                                lr=LR, max_epochs=EPOCHS, patience=PATIENCE, batch_size=BS, seed=seed)
    return tabtransformer_auc(m, fr["Xcat"][te], fr["Xnum"][te], fr["y"][te])

def run_mlp(fr, tr, va, te, seed):
    torch.manual_seed(seed)
    m = TabMLP(fr["Xdense"].shape[1], d_block=128, n_blocks=2, dropout=0.1)
    m, _ = train_net(m, fr["Xdense"][tr], fr["y"][tr], fr["Xdense"][va], fr["y"][va],
                     lr=2e-3, max_epochs=200, patience=16, seed=seed)
    return net_auc(m, fr["Xdense"][te], fr["y"][te])

def run_catboost(fr, tr, va, te, seed):
    from catboost import CatBoostClassifier
    Xdf = fr["Xdf"]; cat_idx = [Xdf.columns.get_loc(c) for c in fr["cat_names"]]
    Xstr = Xdf.copy()
    for c in fr["cat_names"]:
        Xstr[c] = Xstr[c].map(str).astype(object)
    clf = CatBoostClassifier(depth=6, iterations=300, learning_rate=0.05, l2_leaf_reg=3.0,
                             random_seed=seed, thread_count=2, verbose=0, allow_writing_files=False)
    clf.fit(Xstr.iloc[tr], fr["y"][tr].astype(int), cat_features=cat_idx,
            eval_set=(Xstr.iloc[va], fr["y"][va].astype(int)))
    return roc_auc_score(fr["y"][te], clf.predict_proba(Xstr.iloc[te])[:, 1])

MODELS = ["ft_transformer", "tabtransformer", "mlp", "catboost"]
RUN = {"ft_transformer": run_ft, "tabtransformer": run_tabt, "mlp": run_mlp, "catboost": run_catboost}
table = {}
for name in DATASETS:
    fr = load_frame(name)
    print(f"\n=== {name}: {len(fr['y'])} rows | num_frac {fr['num_frac']:.2f} ===", flush=True)
    rows = {m: [] for m in MODELS}
    for s in SEEDS:
        tr, va, te = split_idx(len(fr["y"]), fr["y"], s)
        for m in MODELS:
            # call each model's runner on the SAME split
            rows[m].append(RUN[m](fr, tr, va, te, ____))
        print("  seed {}: ".format(s) + " | ".join(f"{m} {rows[m][-1]:.3f}" for m in MODELS), flush=True)
    table[name] = {m: (float(np.mean(v)), float(np.std(v))) for m, v in rows.items()}
    table[name]["num_frac"] = fr["num_frac"]

# --- cross-dataset summary: mean ranks (1 = best per dataset) + Friedman
score = np.array([[table[d][m][0] for m in MODELS] for d in DATASETS])
ranks = np.array([rankdata(-row, method="average") for row in score])          # HIGHEST score -> rank 1
mean_rank = {m: float(ranks[:, i].mean()) for i, m in enumerate(MODELS)}
fried = friedmanchisquare(*[score[:, i] for i in range(len(MODELS))])
ft_beats_tabt = sum(table[d]["ft_transformer"][0] > table[d]["tabtransformer"][0] for d in DATASETS)
best_neural = min(["ft_transformer", "tabtransformer", "mlp"], key=lambda m: mean_rank[m])

print("\nmean ranks:", {m: round(r, 2) for m, r in mean_rank.items()})
print(f"Friedman chi2={fried.statistic:.3f}, p={fried.pvalue:.3f} (k={len(MODELS)}, N={len(DATASETS)})")
print(f"FT-T beats TabTransformer on {ft_beats_tabt}/{len(DATASETS)}; best neural model = {best_neural}")

In [ ]:
# CHECK — read the verdict the disciplined way (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("ranks are a valid per-dataset ranking of the 4 models",
    ranks.shape == (len(DATASETS), 4) and bool(np.allclose(ranks.sum(1), 10.0)))
chk("best score on each dataset got rank 1",
    all(ranks[i][np.argmax(score[i])] == 1 for i in range(len(DATASETS))))

print(f"""
VERDICT — FT-Transformer mean rank {mean_rank['ft_transformer']:.2f}
          TabTransformer  mean rank {mean_rank['tabtransformer']:.2f}
          MLP             mean rank {mean_rank['mlp']:.2f}
          CatBoost        mean rank {mean_rank['catboost']:.2f}
  FT-T vs TabTransformer : {ft_beats_tabt}/{len(DATASETS)} tables here — the numeric tokenizer should help MOST
                           on the numeric-heavy table (churn, num-frac 0.80).
  best NEURAL model      : {best_neural}
  vs trees               : {'CatBoost still ranks best' if mean_rank['catboost'] <= min(mean_rank[m] for m in ['ft_transformer','tabtransformer','mlp']) else 'a neural model led (unusual at this scale)'}

Read this the disciplined way (NOTES #23). At two tables and two seeds you have little power, so treat the
DIRECTION, not the exact ranks, as the signal. On the lesson's FULLER run (labs/_verify_l046.py: 4 datasets
x 3 seeds) it resolves to mean ranks FT-T 2.50 / MLP 2.75 / TabTransformer 3.75 / CatBoost 1.00 — FT-T beats
TabTransformer on 3/4 (all but the most-categorical credit_g), is the best neural model, and CatBoost wins
all 4 (Friedman p=0.026). FT-Transformer is the strongest single DEEP model AND still loses to the tree —
Gorishniy 2021's exact finding. Do not average the two readings into "FT-T won" or "FT-T lost".""")
print("Task 3", "OK" if ok else "-- fix the FAILs above")

## Task 4 — the probe: do numerics actually attend?

**Goal.** Show the one structural difference between FT-Transformer and TabTransformer *directly*: bump a
single **numeric** feature and measure how much each model's row representation moves.

- **FT-Transformer** reads out the final `[CLS]` vector (`model.cls_readout(x_num, x_cat)`). Because numerics
  are tokens, a numeric change flows through attention into `[CLS]` — the readout should **move**.
- **TabTransformer** builds its representation from the categorical contextual tokens only
  (`model.contextual(x_cat)`); numerics never enter it — so the same numeric change moves it **exactly 0**.

This is the lesson's headline made measurable: FT-Transformer fixes the numeric bypass.

In [ ]:
# TODO — fill every ____.
fr = load_frame("churn")                                    # numeric-heavy (num-frac 0.80)
torch.manual_seed(0)
ft = FTTransformer(fr["Xnum"].shape[1], fr["cards"], **FT_CFG).eval()
tt = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **TT_CFG).eval()

xn, xc = fr["Xnum"][:64], fr["Xcat"][:64]
xn_bumped = xn.clone(); xn_bumped[:, 0] += 3.0              # bump the FIRST numeric feature

with torch.no_grad():
    # FT-Transformer: read out the final [CLS] vector before vs after the numeric bump
    z1 = ft.cls_readout(xn, xc)
    z2 = ft.cls_readout(xn_bumped, xc)
    ft_move = (z1 - z2).norm(dim=1).mean().item()
    # TabTransformer: its representation is the categorical contextual tokens — a numeric bump cannot reach it
    c1 = tt.contextual(xc).flatten(1)
    c2 = ____                                               # recompute the SAME thing (numerics don't enter it)
    tt_move = (c1 - c2).norm(dim=1).mean().item()

print(f"FT-Transformer [CLS] readout move when a numeric feature changes: {ft_move:.3f}")
print(f"TabTransformer contextual move for the same numeric change      : {tt_move:.1e}")

In [ ]:
# CHECK — the numeric-bypass fix, made measurable (do not edit)
ok = True
def chk(name, cond, detail=""):
    global ok
    print(("PASS  " if cond else "FAIL  ") + name + (f"   [{detail}]" if detail else ""))
    if not cond: ok = False

chk("FT-Transformer's [CLS] readout MOVES when a numeric feature changes (numbers attend)",
    ft_move > 1e-4, f"L2 move {ft_move:.3f}")
chk("TabTransformer's representation does NOT move for a numeric change (numerics bypass attention)",
    tt_move < 1e-9, f"L2 move {tt_move:.1e}")
print(f"""
This is FT-Transformer's whole contribution in one number. Because numerics are TOKENS, changing a number
reshapes the row summary ([CLS] move {ft_move:.3f}); in TabTransformer the same change moves the
representation {tt_move:.0e} — the numeric bypass L045 named. The lesson's verify records FT-T 0.438 vs
TabTransformer 0.0 on adult. That structural fix is why FT-T beats TabTransformer most where numerics carry
the signal — and why it, not TabTransformer, is the strongest single neural baseline.""")
print("Task 4", "OK" if ok else "-- fix the FAILs above")

## EXIT TICKET

Paste this output to your teacher, or just say *"lab done."*

In [ ]:
# EXIT TICKET
print("=== LAB 046 — FT-Transformer (feature tokenizer + [CLS] readout) ===")
print(f"numeric tokenizer : affine T_j = b_j + x_j·W_j, validated vs reference (per-feature, order-preserving)")
print(f"[CLS] readout     : prepended at position 0, validated vs reference")
print(f"bake-off ranks    : " + ", ".join(f"{m} {mean_rank[m]:.2f}" for m in MODELS))
print(f"Friedman          : chi2={fried.statistic:.3f}, p={fried.pvalue:.3f} (N={len(DATASETS)} tables)")
print(f"FT-T vs TabT      : FT-Transformer beats TabTransformer on {ft_beats_tabt}/{len(DATASETS)} tables")
print(f"numerics attend   : FT-T [CLS] move {ft_move:.3f} vs TabTransformer {tt_move:.0e} (the bypass, fixed)")
print()
print("in one sentence, what does tokenising the numeric features buy over TabTransformer, and does it beat a tree?:", "____")

## NEXT STEP — reproduce the paper's results (required, not stretch)

The EXIT ticket above is the **learning lab**: you implemented the architecture and ran a
downscaled bake-off that fits in minutes on CPU. That is a *different experiment* from the
paper's table. **Do not treat the EXIT ranking as the paper's result.** Mixing those two
buckets is how you learn the wrong conclusion (standard #25 / M60).

**Paper:** Gorishniy, Rubachev, Khrulkov & Babenko 2021, FT-Transformer ([arXiv:2106.11959](https://arxiv.org/abs/2106.11959))

**Bucket 1 — verified here (this notebook's budget)**
- **mean ranks:** FT-T 2.50 vs MLP 2.75 vs TabTransformer 3.75 vs CatBoost 1.00, Friedman p=0.026
- **numeric-bypass fix:** FT-T beats TabTransformer 3/4 (all but the most-categorical table)

**Bucket 2 — the paper's claim (cited, not yet reproduced by this notebook)**
- **FT-T is the strongest single deep model:** 11-dataset tuned mean. We use a random OpenML split + no Optuna → INCOMPARABLE on Table 2; read DIRECTION vs MLP / TabTransformer.
- **ties tuned GBDTs:** Paper: FT-T ~ties GBDT on average. On Adult a tree edges it — CatBoost winning here is compatible.
- **Adult accuracy ≈ 0.859 (tuned):** Table 2 default/tuned FT-T; our untuned random-split number is INCOMPARABLE.

**Bucket 3 — scale-up run (you train this).** Same from-scratch code, closer to the paper's
dataset / budget / metric. Two operators:

1. **Google Colab (you, GPU).** `Runtime → Change runtime type → T4 GPU`, set
   `RUN_PAPER_REPRO = True` in the next cell, run it. Stay in the tab — free Colab
   disconnects after ~90 min of no *tab* interaction, even if training is still going.
2. **Modal (unattended).** From the repo root:
   ```
   ~/.local/bin/modal run --detach modal/l046_paper_repro.py --preset closer
   ```
   Use `--preset paper` only when you can spend hours and want paper hyperparameters.
   `smoke` is a seconds-long import check, not a result.

When it finishes, the cell prints a **ledger** with MATCH / CLOSE / FAIL / INCOMPARABLE /
DIRECTION_*. Paste that ledger to your teacher. Until you run it, the paper claim stays
*cited, not reproduced* — and that is an honest state, not a failure.


In [ ]:
# PROVIDED — paper-results scale-up, inlined from `labs/_paper_repro_l046.py`.
# Read this cell: it is the training / comparison loop, not a hidden package.
# Default OFF so the learning lab stays minutes. On Colab: Runtime → T4 GPU,
# set RUN_PAPER_REPRO = True, re-run. Unattended: modal run --detach modal/l046_paper_repro.py

"""L046 paper-results scale-up (NOTES standard #25).

The learning lab trains FT-Transformer on small OpenML tables at a tiny budget. Gorishniy et al. 2021
(FT-Transformer, arXiv:2106.11959, Table 2) report — under a *tuned*, unified protocol on 11 datasets —
that FT-Transformer is the **strongest single deep model** and **roughly ties tuned GBDTs**, while
consistently **beating an MLP and TabTransformer**. On some tables (e.g. Adult) GBDTs still edge it.

This harness re-runs the same from-scratch FT-Transformer on a **paper dataset** (Adult, OpenML 1590; or
Higgs-small, OpenML 23512) against TabTransformer (L045), an MLP, and CatBoost, at three budgets. Absolute
Table-2 numbers stay INCOMPARABLE (we use a random OpenML split and do not run the paper's Optuna tuning);
the DIRECTION tests are what can actually update belief:

  * FT-T vs TabTransformer   (paper: FT-T better — tokenising *numerics* is the upgrade)
  * FT-T vs MLP              (paper: FT-T ≥ MLP — the strongest single deep model)
  * FT-T vs CatBoost         (paper: ~tie on average; GBDT can still win a given table)

Presets: smoke · closer · paper.

Run:
    OMP_NUM_THREADS=1 python labs/_paper_repro_l046.py --preset smoke
    ~/.local/bin/modal run --detach modal/l046_paper_repro.py --preset closer
"""
from __future__ import annotations

import argparse
import json
import os
import sys
import time
import warnings

warnings.filterwarnings("ignore")
os.environ.setdefault("OMP_NUM_THREADS", "1")

import numpy as np
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score, roc_auc_score
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in globals() else os.getcwd()
sys.path.insert(0, HERE)

from relkit import load_tier_a  # noqa: E402
from relkit.paper_repro import (  # noqa: E402
    LabFinding, PaperTarget, ScaleUpRun,
    classify_direction, classify_number, device, format_ledger, hardware_tag,
    print_howto, to_jsonable,
)
from relkit.tabtransformer import (  # noqa: E402
    TabTransformer, frame_categorical, tabtransformer_auc, train_tabtransformer,
)
from relkit.nets import TabMLP, net_auc, train_net  # noqa: E402

# Gorishniy 2021, Table 2 — Adult, accuracy. FT-T is the top DL model but GBDTs edge it on THIS table.
ADULT_TARGET = PaperTarget(
    paper="FT-Transformer (Gorishniy et al. 2021)", arxiv="2106.11959",
    table="Table 2 — Adult, accuracy (tuned)",
    dataset="adult (full)", metric="accuracy", paper_value=0.859, abs_tol=0.01,
    paper_split="paper: fixed train/val/test + Optuna tuning; we use OpenML 1590 random split, no tuning",
    higher_is_better=True,
    notes="Table 2 default/tuned FT-T ≈ 0.859; CatBoost/XGBoost ≈ 0.873 (GBDT edges FT-T on Adult).",
)
VS_TABT_TARGET = PaperTarget(
    paper="FT-Transformer (Gorishniy et al. 2021)", arxiv="2106.11959",
    table="Table 2 — FT-T > TabTransformer (tokenising numerics)",
    dataset="adult (full)", metric="auc_minus_tabtransformer", paper_value=0.0, abs_tol=0.02,
    notes="Paper: FT-T beats TabTransformer; the mechanism is that numerics are tokenised and attend.",
)
VS_MLP_TARGET = PaperTarget(
    paper="FT-Transformer (Gorishniy et al. 2021)", arxiv="2106.11959",
    table="Table 2 — FT-T is the strongest single deep model (≥ MLP)",
    dataset="adult (full)", metric="auc_minus_mlp", paper_value=0.0, abs_tol=0.02,
    notes="Paper: FT-T ≥ MLP across the benchmark (often close on any single easy table).",
)

LAB_FINDINGS = [
    LabFinding("FT-T vs TabTransformer vs MLP vs CatBoost mean ranks",
               "see labs/_verify_l046_results.json (4 small tables, 3 seeds, CPU)",
               "credit_g + adult(4000) + churn + phoneme — a demonstration, not the 11-dataset benchmark"),
    LabFinding("numeric-bypass fix",
               "FT-T beats TabTransformer most where numerics carry the signal (num_frac high)",
               "mechanism validated exactly; direction reproduced at small scale"),
]


def load_frame(name, *, cap=None, seed=0):
    Xdf, y = load_tier_a(name)
    if cap is not None and len(Xdf) > cap:
        idx, _ = train_test_split(np.arange(len(Xdf)), train_size=cap, random_state=seed, stratify=y)
        Xdf, y = Xdf.iloc[idx].reset_index(drop=True), y.iloc[idx].reset_index(drop=True)
    # one incomplete OpenML row (e.g. higgs_small) breaks StandardScaler → drop NaN-feature rows
    keep = Xdf.notna().all(axis=1)
    if not bool(keep.all()):
        Xdf, y = Xdf.loc[keep].reset_index(drop=True), y.loc[keep].reset_index(drop=True)
    Xcat, Xnum, cards, cat_names, num_names = frame_categorical(Xdf)
    ct = ColumnTransformer([
        ("num", StandardScaler(), num_names),
        ("cat", OneHotEncoder(handle_unknown="ignore", sparse_output=False), cat_names),
    ])
    Xdense = ct.fit_transform(Xdf).astype(np.float32)
    return dict(Xdf=Xdf, Xcat=Xcat, Xnum=Xnum, cards=cards, y=y.to_numpy().astype(np.float32),
                cat_names=cat_names, num_names=num_names, Xdense=Xdense,
                num_frac=len(num_names) / max(len(num_names) + len(cat_names), 1))


def split_idx(n, y, seed):
    tr, te = train_test_split(np.arange(n), test_size=0.30, random_state=seed, stratify=y)
    tr, va = train_test_split(tr, test_size=0.25, random_state=seed, stratify=y[tr])
    return tr, va, te


def _acc(p, y):
    return float(accuracy_score(y, (np.asarray(p) >= 0.5).astype(int)))


def run_ft(fr, tr, va, te, *, cfg, epochs, lr, seed, dev):
    m = FTTransformer(fr["Xnum"].shape[1], fr["cards"], **cfg)
    t0 = time.time()
    m, _ = train_ft_transformer(m, fr["Xnum"][tr], fr["Xcat"][tr], fr["y"][tr],
                                fr["Xnum"][va], fr["Xcat"][va], fr["y"][va],
                                lr=lr, max_epochs=epochs, patience=max(6, epochs // 4),
                                batch_size=256, device=dev, seed=seed)
    import torch
    with torch.no_grad():
        xn = torch.as_tensor(np.asarray(fr["Xnum"][te]), dtype=torch.float32, device=dev)
        xc = torch.as_tensor(np.asarray(fr["Xcat"][te]), dtype=torch.long, device=dev)
        p = torch.sigmoid(m(xn, xc)).cpu().numpy()
    return {"auc": float(roc_auc_score(fr["y"][te], p)), "acc": _acc(p, fr["y"][te])}, time.time() - t0


def run_tabt(fr, tr, va, te, *, epochs, lr, seed, dev):
    cfg = dict(d=32, n_layers=3, n_heads=4, head_hidden=128, dropout=0.1)
    m = TabTransformer(fr["cards"], fr["Xnum"].shape[1], **cfg)
    m, _ = train_tabtransformer(m, fr["Xcat"][tr], fr["Xnum"][tr], fr["y"][tr],
                                fr["Xcat"][va], fr["Xnum"][va], fr["y"][va],
                                lr=lr, max_epochs=epochs, patience=max(6, epochs // 4),
                                batch_size=256, device=dev, seed=seed)
    return float(tabtransformer_auc(m, fr["Xcat"][te], fr["Xnum"][te], fr["y"][te], device=dev))


def run_mlp(fr, tr, va, te, *, epochs, seed):
    m = TabMLP(fr["Xdense"].shape[1], d_block=128, n_blocks=2, dropout=0.1)
    m, _ = train_net(m, fr["Xdense"][tr], fr["y"][tr], fr["Xdense"][va], fr["y"][va],
                     lr=2e-3, max_epochs=epochs, patience=16, seed=seed)
    return float(net_auc(m, fr["Xdense"][te], fr["y"][te]))


def run_catboost(fr, tr, va, te, *, seed):
    from catboost import CatBoostClassifier
    Xdf = fr["Xdf"]
    cat_idx = [Xdf.columns.get_loc(c) for c in fr["cat_names"]]
    Xstr = Xdf.copy()
    for c in fr["cat_names"]:
        Xstr[c] = Xstr[c].map(str).astype(object)
    clf = CatBoostClassifier(iterations=1000, depth=6, learning_rate=0.05, verbose=0,
                             random_seed=seed, allow_writing_files=False)
    clf.fit(Xstr.iloc[tr], fr["y"][tr].astype(int), cat_features=cat_idx,
            eval_set=(Xstr.iloc[va], fr["y"][va].astype(int)), use_best_model=True)
    p = clf.predict_proba(Xstr.iloc[te])[:, 1]
    return {"auc": float(roc_auc_score(fr["y"][te], p)), "acc": _acc(p, fr["y"][te])}


def preset_cfg(name):
    if name == "smoke":
        return dict(cap=700, epochs=2, ft=dict(d=32, n_layers=1, n_heads=4, dropout=0.1))
    if name == "closer":
        return dict(cap=None, epochs=30, ft=dict(d=64, n_layers=3, n_heads=8, dropout=0.1))
    if name == "paper":
        # Gorishniy default FT-T is d≈192, 3 layers, 8 heads. Heavier; T4/paper budget.
        return dict(cap=None, epochs=50, ft=dict(d=192, n_layers=3, n_heads=8, dropout=0.1))
    raise ValueError(f"unknown preset {name!r}")


def main(argv=None):
    p = argparse.ArgumentParser()
    p.add_argument("--preset", choices=("smoke", "closer", "paper"), default="closer")
    p.add_argument("--dataset", choices=("adult", "higgs_small"), default="adult")
    args = p.parse_args(argv)
    cfg = preset_cfg(args.preset)
    dev = device()
    hw = hardware_tag()
    print(f"L046 paper-repro  preset={args.preset}  dataset={args.dataset}  device={hw}")

    fr = load_frame(args.dataset, cap=cfg["cap"])
    tr, va, te = split_idx(len(fr["y"]), fr["y"], 0)
    print(f"  {args.dataset} n={len(fr['y'])}  cats={len(fr['cards'])}  nums={fr['Xnum'].shape[1]}  "
          f"num_frac={fr['num_frac']:.2f}")

    ft_res = ft_run = tabt_auc = mlp_auc = cb_res = None
    try:
        ft_res, ft_wall = run_ft(fr, tr, va, te, cfg=cfg["ft"], epochs=cfg["epochs"], lr=1e-3, seed=0, dev=dev)
        tabt_auc = run_tabt(fr, tr, va, te, epochs=cfg["epochs"], lr=1e-3, seed=0, dev=dev)
        mlp_auc = run_mlp(fr, tr, va, te, epochs=max(cfg["epochs"] * 3, 60), seed=0)
        cb_res = run_catboost(fr, tr, va, te, seed=0)
        ft_run = ScaleUpRun(
            method="ft-transformer-scratch", dataset=args.dataset, metric="accuracy",
            value=ft_res["acc"], n_seeds=1, hardware=hw, wall_s=ft_wall, protocol_match=False,
            protocol_deviations=[
                f"single dataset ({args.dataset}), not the paper's 11-dataset tuned mean",
                "OpenML random split + no Optuna tuning (paper tunes every model)",
                f"FT-T auc={ft_res['auc']:.4f}  TabT auc={tabt_auc:.4f}  "
                f"MLP auc={mlp_auc:.4f}  CatBoost auc={cb_res['auc']:.4f} acc={cb_res['acc']:.4f}",
            ],
        )
        print(f"  FT-T acc={ft_res['acc']:.4f} auc={ft_res['auc']:.4f} | TabT auc={tabt_auc:.4f} | "
              f"MLP auc={mlp_auc:.4f} | CatBoost acc={cb_res['acc']:.4f} auc={cb_res['auc']:.4f} "
              f"| wall={ft_wall:.0f}s")
    except Exception as exc:
        import traceback
        print(f"  bake-off failed: {exc}")
        traceback.print_exc()

    extra = []
    if ft_res is not None:
        extra.append(
            f"DIRECTION FT-T vs TabTransformer (auc): "
            f"{classify_direction(ft_res['auc'], tabt_auc, paper_a_beats_b=True, tie_tol=0.005)} "
            f"(paper: FT-T beats TabTransformer by tokenising numerics)."
        )
        extra.append(
            f"DIRECTION FT-T vs MLP (auc): "
            f"{classify_direction(ft_res['auc'], mlp_auc, paper_a_beats_b=True, tie_tol=0.005)} "
            f"(paper: FT-T is the strongest single deep model, ≥ MLP)."
        )
        extra.append(
            f"DIRECTION FT-T vs CatBoost (auc): "
            f"{classify_direction(ft_res['auc'], cb_res['auc'], paper_a_beats_b=False, tie_tol=0.005)} "
            f"(paper: FT-T ~ties tuned GBDTs on average; on Adult, GBDT edges it — a loss is compatible)."
        )

    delta_tabt = None if ft_res is None else ScaleUpRun(
        method="delta", dataset=args.dataset, metric="auc_minus_tabtransformer",
        value=ft_res["auc"] - tabt_auc, n_seeds=1, hardware=hw,
        protocol_match=False, protocol_deviations=ft_run.protocol_deviations)
    delta_mlp = None if ft_res is None else ScaleUpRun(
        method="delta", dataset=args.dataset, metric="auc_minus_mlp",
        value=ft_res["auc"] - mlp_auc, n_seeds=1, hardware=hw,
        protocol_match=False, protocol_deviations=ft_run.protocol_deviations)

    rows = [
        (ADULT_TARGET, ft_run, classify_number(ADULT_TARGET, ft_run)),
        (VS_TABT_TARGET, delta_tabt, classify_number(VS_TABT_TARGET, delta_tabt)),
        (VS_MLP_TARGET, delta_mlp, classify_number(VS_MLP_TARGET, delta_mlp)),
    ]
    text = format_ledger(title="L046 FT-Transformer", lab=LAB_FINDINGS, paper=rows, extra_lines=extra)
    print()
    print(text)

    out = {
        "lesson": 46, "preset": args.preset, "dataset": args.dataset, "hardware": hw,
        "ft": to_jsonable(ft_run), "ft_auc": None if ft_res is None else ft_res["auc"],
        "tabtransformer_auc": tabt_auc, "mlp_auc": mlp_auc,
        "catboost": cb_res, "ledger": text,
    }
    dest = os.environ.get("PAPER_REPRO_OUT") or os.path.join(
        HERE, f"_paper_repro_l046_{args.preset}_results.json"
    )
    with open(dest, "w") as fh:
        json.dump(out, fh, indent=2)
    print(f"\nwrote {dest}")
    return out

from relkit.paper_repro import print_howto

RUN_PAPER_REPRO = False
PRESET = "closer"          # smoke | closer | paper

if RUN_PAPER_REPRO:
    main(["--preset", PRESET])
else:
    print_howto(lesson=46, modal="modal/l046_paper_repro.py", harness="labs/_paper_repro_l046.py")


## Stretch (optional, ungraded) — after the scale-up

1. **The numeric-fraction sweep.** Add `phoneme` (all numeric) and `credit_g` (mostly categorical) to
   `DATASETS` and re-run Task 3. FT-Transformer's edge over TabTransformer should *grow* with the numeric
   fraction — and vanish (or invert) on the most categorical table. This is the mechanism, plotted.
2. **How wide a token?** Sweep `d ∈ {16, 32, 64, 128}` in `FT_CFG` on `churn`. Where do returns flatten? The
   paper's default is ~192, far above what a small table needs.
3. **Does [CLS] beat mean-pooling?** Replace the [CLS] readout with the mean of the feature tokens (edit a
   copy of `cls_readout`) and compare. How much does the learned readout actually buy on these tables?
4. **Depth.** Sweep `n_layers ∈ {1, 2, 3}`. On small tables, does one attention layer already capture most of
   the cross-feature signal?

In [ ]:
# STRETCH — ungraded.
# for extra in (["phoneme"], ["credit_g"]):
#     for name in extra:
#         fr = load_frame(name)
#         accs_ft, accs_tt = [], []
#         for s in SEEDS:
#             tr, va, te = split_idx(len(fr["y"]), fr["y"], s)
#             accs_ft.append(run_ft(fr, tr, va, te, s)); accs_tt.append(run_tabt(fr, tr, va, te, s))
#         print(f"{name} (num_frac {fr['num_frac']:.2f}): FT-T {np.mean(accs_ft):.3f}  TabT {np.mean(accs_tt):.3f}")